In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del archivo genre.csv

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
genre_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/genre.csv")
) 
display(genre_df)

In [0]:
genre_schema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("genreName", StringType(), True)
])
genre_df = (
    spark.read
        .format('csv')
        .schema(genre_schema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/genre.csv")
)
display(genre_df)
genre_df.printSchema()

In [0]:
genre_df_final = (
    add_ingestion_date(genre_df)
    .select(
        col("genreId").alias("genre_id"),
        col("genreName").alias("genre_name"),
        lit(v_env).alias("env"),
        lit(v_file_date).alias("file_date")
    )
)
display(genre_df_final)

In [0]:
(genre_df_final.write
    .mode("overwrite")
    .saveAsTable("movie_silver.genres")
)

In [0]:
dbutils.notebook.exit("Success")